# Object Detection: from IoU to mAP with YOLO

**Module 01 · Perception Systems** · Notebook 4 of 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/01_Perception/04_object_detection.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~75 min | [02 Sensor modalities](02_sensor_modalities.ipynb); basic Python/NumPy | ISO 21448:2022; ISO/PAS 8800:2024 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Describe what a 2D object detector outputs and how one-stage, two-stage and transformer detectors differ
2. Run a pretrained YOLO model on a street scene and interpret boxes, classes and confidence scores
3. Implement Intersection over Union (IoU) and Non-Maximum Suppression (NMS) from scratch
4. Compute precision, recall, the precision–recall curve and Average Precision (AP, mAP@[.5:.95]) from scratch
5. Distinguish throughput (FPS) from latency and relate both to the driving task
6. Explain why detector confidence is not a calibrated probability and why that matters for safety

## Setup

This notebook uses the [Ultralytics](https://docs.ultralytics.com/) YOLO package when it is available. On **Google Colab** it is installed automatically. Locally, install it yourself (`pip install ultralytics`) — the notebook does not change your environment because installing Ultralytics can also replace an existing PyTorch/torchvision build.

**Without Ultralytics, or without network access, the notebook still runs top to bottom:** it falls back to a stored, clearly labelled detector output so that all the teaching cells (IoU, NMS, precision/recall, AP) work.

In [ ]:
import sys, importlib.util, subprocess, time, json
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and importlib.util.find_spec("ultralytics") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=False)

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except Exception as e:                      # ImportError or a broken torch/torchvision combination
    YOLO_AVAILABLE = False
    print(f"Ultralytics not available ({type(e).__name__}); using the stored fallback detections.")

rng = np.random.default_rng(42)
OKABE_ITO = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']
plt.rcParams.update({'figure.dpi': 100})
print(f"NumPy {np.__version__} | Ultralytics available: {YOLO_AVAILABLE}")

---
## 1. What is object detection?

**Input:** an image (e.g. 1920 × 1080 pixels, 3 colour channels).
**Output:** a list of objects, each with
- a **bounding box** — e.g. `[x1, y1, x2, y2]` (corner format, used below) or `[x, y, w, h]` (COCO format: top-left corner plus width and height);
- a **class label** — "car", "person", "bicycle", …;
- a **confidence score** in [0, 1].

```python
[{"class": "car",    "bbox": [100, 200, 250, 280], "confidence": 0.95},
 {"class": "person", "bbox": [500, 300, 550, 420], "confidence": 0.87}]
```

### Architecture families
- **Two-stage** (R-CNN family, e.g. Faster R-CNN): region proposals, then classification/refinement. The original Faster R-CNN paper reported about 5 fps on 2015 GPU hardware.
- **One-stage** (SSD, YOLO family): dense prediction in a single pass; real-time on modern GPUs. Frame rates are hardware-dependent — always measure on your target.
- **Transformer-based** (DETR, RT-DETR): set prediction without hand-designed anchors; RT-DETR is real-time.

**YOLO versions (as of October 2026):** YOLOv8 (2023) is a well-documented baseline; YOLO11 (2024) and YOLO26 (released January 2026, end-to-end **NMS-free**) offer better speed/accuracy trade-offs ([Ultralytics docs](https://docs.ultralytics.com/compare/yolo26-vs-yolo11)). The API is identical (`YOLO('yolo11n.pt')`). This notebook uses **YOLO11n** ("nano"), and compares variants in the benchmark section.

All models here are trained on **COCO** (80 everyday classes, including person, car, truck, bus, bicycle, motorcycle, traffic light, stop sign). COCO is *not* a driving dataset; production systems are trained on driving data (see [08 Driving datasets](08_driving_datasets.ipynb)).

---
## 2. Run a detector on a street scene

We use the Ultralytics sample image `bus.jpg` (a bus and pedestrians on a street). If it cannot be downloaded, a plain placeholder canvas is used instead.

In [ ]:
IMAGE_URL = "https://ultralytics.com/images/bus.jpg"

def load_image(url, timeout=15):
    """Download an image as an RGB uint8 array; return None on any failure."""
    try:
        import requests
        from PIL import Image
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        return np.array(Image.open(BytesIO(r.content)).convert('RGB'))
    except Exception as e:
        print(f"Image download failed ({type(e).__name__}); using a placeholder canvas.")
        return None

img = load_image(IMAGE_URL)
IMAGE_IS_REAL = img is not None
if not IMAGE_IS_REAL:
    img = np.full((1080, 810, 3), 200, np.uint8)       # same size as bus.jpg
print(f"Image shape (H, W, C): {img.shape}; real photo: {IMAGE_IS_REAL}")

In [ ]:
# Stored output of YOLO11n (Ultralytics 8.4.191, CPU, conf >= 0.05) on bus.jpg.
# Used ONLY when the model or image cannot be loaded, so that all later cells still run.
FALLBACK_DETECTIONS = [
    ["bus", 0.94, [3.8, 229.4, 796.2, 728.4]], ["person", 0.888, [671.0, 394.8, 809.8, 878.7]],
    ["person", 0.878, [47.4, 399.6, 239.3, 904.2]], ["person", 0.856, [223.1, 408.7, 344.5, 860.4]],
    ["person", 0.622, [0.0, 556.1, 68.9, 872.4]], ["skateboard", 0.119, [668.7, 822.7, 809.4, 884.4]],
    ["fire hydrant", 0.092, [655.5, 622.2, 691.9, 715.7]], ["person", 0.092, [0.2, 601.3, 49.9, 870.6]],
    ["fire hydrant", 0.071, [423.3, 659.9, 461.3, 763.4]], ["handbag", 0.064, [101.4, 465.6, 203.1, 604.7]],
    ["stop sign", 0.059, [0.2, 250.6, 33.7, 323.9]],
]

MODEL_NAME = 'yolo11n.pt'
model = None
detections, SOURCE = None, None
if YOLO_AVAILABLE and IMAGE_IS_REAL:
    try:
        model = YOLO(MODEL_NAME)                                   # downloads ~5 MB weights on first use
        t0 = time.perf_counter()
        res = model.predict(np.ascontiguousarray(img[..., ::-1]),  # Ultralytics expects BGR arrays
                            conf=0.05, verbose=False)[0]
        latency_ms = (time.perf_counter() - t0) * 1000
        detections = [[model.names[int(c)], float(s), [float(v) for v in b]]
                      for c, s, b in zip(res.boxes.cls, res.boxes.conf, res.boxes.xyxy)]
        SOURCE = f"live {MODEL_NAME} inference ({latency_ms:.0f} ms incl. first-call overhead)"
    except Exception as e:
        print(f"Model load/inference failed ({type(e).__name__}: {str(e)[:80]}); using stored detections.")
if detections is None:
    detections = FALLBACK_DETECTIONS
    SOURCE = "STORED fallback output of YOLO11n (not computed in this session)"

det_df = pd.DataFrame([{'class': c, 'confidence': round(s, 3), 'x1': b[0], 'y1': b[1], 'x2': b[2], 'y2': b[3]}
                       for c, s, b in detections]).sort_values('confidence', ascending=False, ignore_index=True)
print(f"Detections source: {SOURCE}")
display(det_df)

In [ ]:
CLASS_COLOURS = {'person': OKABE_ITO[5], 'car': OKABE_ITO[4], 'bus': OKABE_ITO[2], 'truck': OKABE_ITO[2],
                 'bicycle': OKABE_ITO[0], 'motorcycle': OKABE_ITO[0]}

def draw(ax, image, dets, conf_thr=0.25, title=''):
    ax.imshow(image); ax.axis('off')
    kept = [d for d in dets if d[1] >= conf_thr]
    for cls, conf, (x1, y1, x2, y2) in kept:
        col = CLASS_COLOURS.get(cls, OKABE_ITO[1])
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2.5, edgecolor=col))
        ax.text(x1 + 3, y1 + 18, f"{cls} {conf:.2f}", color='white', fontsize=9,
                bbox=dict(facecolor=col, alpha=0.85, pad=1, edgecolor='none'))
    ax.set_title(f"{title}\n{len(kept)} detections with confidence >= {conf_thr}")
    return kept

fig, ax = plt.subplots(figsize=(7, 9))
kept = draw(ax, img, detections, 0.25, f"Detections ({'live' if model else 'stored'})")
plt.tight_layout(); plt.show()

print("--- Detection summary (text) ---")
for cls, conf, b in kept:
    print(f"  {cls:10s} conf {conf:.3f}  box [{b[0]:.0f}, {b[1]:.0f}, {b[2]:.0f}, {b[3]:.0f}]")
counts = pd.Series([d[0] for d in kept]).value_counts()
print("  Classes found: " + ", ".join(f"{k} x{v}" for k, v in counts.items()))

---
## 3. Intersection over Union (IoU)

IoU measures how well two boxes overlap:

$$ \mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|} $$

It is used (i) to decide whether a detection matches a ground-truth object (e.g. IoU ≥ 0.5) and (ii) inside NMS to decide whether two detections are duplicates.

In [ ]:
def iou(a, b):
    """IoU of two boxes in [x1, y1, x2, y2] format."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def iou_matrix(A, B):
    """Vectorised IoU between every box in A (N,4) and B (M,4)."""
    A, B = np.asarray(A, float), np.asarray(B, float)
    ix1 = np.maximum(A[:, None, 0], B[None, :, 0]); iy1 = np.maximum(A[:, None, 1], B[None, :, 1])
    ix2 = np.minimum(A[:, None, 2], B[None, :, 2]); iy2 = np.minimum(A[:, None, 3], B[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area = lambda X: (X[:, 2] - X[:, 0]) * (X[:, 3] - X[:, 1])
    union = area(A)[:, None] + area(B)[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)

# Sanity checks with hand-computed values
assert iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1.0
assert iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0.0
assert abs(iou([0, 0, 10, 10], [5, 0, 15, 10]) - 50 / 150) < 1e-12     # half overlap -> 1/3
assert np.allclose(iou_matrix([[0, 0, 10, 10]], [[5, 0, 15, 10], [0, 0, 10, 10]]), [[1/3, 1.0]])

# How fast IoU drops when a box is shifted: a 60 x 150 px pedestrian box shifted horizontally
ped = [100, 100, 160, 250]
shifts = np.arange(0, 61, 5)
ious = [iou(ped, [ped[0] + s, ped[1], ped[2] + s, ped[3]]) for s in shifts]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(shifts, ious, 'o-', color=OKABE_ITO[4])
ax.axhline(0.5, color=OKABE_ITO[5], ls='--', label='IoU = 0.5 (common match threshold)')
ax.set_xlabel('Horizontal shift (px) of a 60 px wide box'); ax.set_ylabel('IoU')
ax.set_title('IoU vs localisation error'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
max_shift = shifts[np.array(ious) >= 0.5].max()
print("IoU checks passed.")
print(f"A 60 px wide box still matches at IoU >= 0.5 for shifts up to {max_shift} px "
      f"({100*max_shift/60:.0f} % of its width): narrow objects such as pedestrians are sensitive to localisation error.")

---
## 4. Non-Maximum Suppression (NMS)

A detector proposes many overlapping boxes for the same object. **Greedy NMS** keeps the highest-scoring box, removes every remaining box of the same class whose IoU with it exceeds a threshold, and repeats. (YOLO26 and DETR-style detectors are trained to output one box per object and do not need this step.)

To see NMS at work we create **synthetic duplicate candidates** around the detections above (jittered copies with lower scores) — this mimics the raw output of a detector before NMS.

> **Illustrative data:** the duplicate candidate boxes are synthetic, chosen for teaching.

In [ ]:
def nms(boxes, scores, iou_thr=0.5):
    """Greedy NMS. Returns indices of kept boxes, in descending score order."""
    boxes, scores = np.asarray(boxes, float), np.asarray(scores, float)
    order = np.argsort(-scores, kind='stable')
    keep = []
    while order.size:
        i = order[0]; keep.append(int(i))
        if order.size == 1: break
        ov = iou_matrix(boxes[i:i + 1], boxes[order[1:]])[0]
        order = order[1:][ov <= iou_thr]
    return keep

def batched_nms(boxes, scores, classes, iou_thr=0.5):
    """Class-aware NMS: boxes of different classes never suppress each other."""
    keep = []
    for c in np.unique(classes):
        idx = np.where(np.asarray(classes) == c)[0]
        keep += [int(idx[k]) for k in nms(np.asarray(boxes)[idx], np.asarray(scores)[idx], iou_thr)]
    return sorted(keep, key=lambda k: -scores[k])

base = [d for d in detections if d[1] >= 0.25]
cand_boxes, cand_scores, cand_cls = [], [], []
for cls, conf, b in base:
    w, h = b[2] - b[0], b[3] - b[1]
    cand_boxes.append(b); cand_scores.append(conf); cand_cls.append(cls)
    for _ in range(6):                                   # 6 jittered duplicates per object
        j = rng.normal(0, 0.06, 4) * np.array([w, h, w, h])
        cand_boxes.append(list(np.array(b) + j)); cand_scores.append(conf * rng.uniform(0.5, 0.98)); cand_cls.append(cls)
cand_boxes = np.array(cand_boxes); cand_scores = np.array(cand_scores); cand_cls = np.array(cand_cls)
keep = batched_nms(cand_boxes, cand_scores, cand_cls, iou_thr=0.5)

# Cross-check against torchvision's reference implementation (if available)
try:
    import torch, torchvision
    tv_keep = []
    for c in np.unique(cand_cls):
        idx = np.where(cand_cls == c)[0]
        k = torchvision.ops.nms(torch.tensor(cand_boxes[idx]), torch.tensor(cand_scores[idx]), 0.5).numpy()
        tv_keep += list(idx[k])
    print(f"Matches torchvision.ops.nms: {sorted(keep) == sorted(int(k) for k in tv_keep)}")
except Exception as e:
    print(f"torchvision cross-check skipped ({type(e).__name__}).")

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
for ax, idxs, ttl in [(axes[0], range(len(cand_boxes)), f'Before NMS: {len(cand_boxes)} candidates'),
                      (axes[1], keep, f'After NMS (IoU 0.5): {len(keep)} boxes')]:
    ax.imshow(img); ax.axis('off'); ax.set_title(ttl)
    for k in idxs:
        x1, y1, x2, y2 = cand_boxes[k]
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=1.2,
                               edgecolor=CLASS_COLOURS.get(cand_cls[k], OKABE_ITO[1])))
plt.tight_layout(); plt.show()
print(f"NMS reduced {len(cand_boxes)} candidates to {len(keep)} boxes for {len(base)} objects "
      f"({'one box per object' if len(keep) == len(base) else 'check: count differs from object count'}).")

**Safety note:** NMS can also delete *true* objects — two pedestrians walking close together, or a child next to an adult, produce highly overlapping boxes. The NMS IoU threshold is therefore a safety-relevant design parameter, and crowded scenes are a known triggering condition.

---
## 5. The confidence threshold

In [ ]:
thresholds = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9]
rows = []
for t in thresholds:
    k = [d for d in detections if d[1] >= t]
    rows.append({'threshold': t, 'detections': len(k), 'persons': sum(d[0] == 'person' for d in k),
                 'classes': ', '.join(sorted(set(d[0] for d in k)))})
thr_df = pd.DataFrame(rows); display(thr_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 6.5))
for ax, t in zip(axes, [0.1, 0.5, 0.85]):
    draw(ax, img, detections, t, f'threshold {t}')
plt.tight_layout(); plt.show()

print("--- Threshold sweep (text summary) ---")
for r in rows:
    print(f"  conf >= {r['threshold']:<4}: {r['detections']:2d} detections ({r['persons']} persons) -> {r['classes']}")
low_conf_persons = [d for d in detections if d[0] == 'person' and d[1] < 0.75]
print(f"  Persons with confidence below 0.75: {len(low_conf_persons)} "
      f"(e.g. partially visible at the image border) -> a high threshold would discard them.")
print("  Lower threshold = more detections (higher recall, lower precision); higher threshold = the reverse.")

---
## 6. Precision, recall and Average Precision (AP)

For a set of images with ground truth (GT):

- a detection is a **true positive (TP)** if it matches a not-yet-matched GT object of the same class with IoU ≥ threshold; otherwise a **false positive (FP)**; unmatched GT objects are **false negatives (FN)**;
- $\text{precision} = TP / (TP + FP)$, $\text{recall} = TP / (TP + FN)$.

Sorting all detections by confidence and sweeping the threshold yields the **precision–recall (PR) curve**. **AP** is the area under the (monotonically interpolated) PR curve:

- *PASCAL VOC (2010+)*: all-point interpolation, $AP = \sum_k (r_k - r_{k-1})\,\max_{r \ge r_k} p(r)$;
- *COCO*: the interpolated precision averaged at 101 recall points, then averaged over IoU thresholds 0.50:0.05:0.95 → **AP@[.5:.95]** (often just called "mAP" when averaged over classes).

In [ ]:
def match_detections(dets, gts, iou_thr):
    """dets: list of (image_id, score, box); gts: dict image_id -> (M,4) array.
    Greedy matching in descending score order. Returns (scores, tp flags, number of GT)."""
    dets = sorted(dets, key=lambda d: -d[1])
    used = {k: np.zeros(len(v), bool) for k, v in gts.items()}
    tp = np.zeros(len(dets), bool)
    for n, (im, s, b) in enumerate(dets):
        g = gts.get(im, np.zeros((0, 4)))
        if len(g) == 0: continue
        ov = iou_matrix([b], g)[0]
        ov[used[im]] = -1                                  # each GT can be matched only once
        j = int(np.argmax(ov))
        if ov[j] >= iou_thr:
            tp[n] = True; used[im][j] = True
    return np.array([d[1] for d in dets]), tp, sum(len(v) for v in gts.values())

def pr_curve(tp, n_gt):
    ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
    recall = ctp / max(n_gt, 1)
    precision = ctp / np.maximum(ctp + cfp, 1)
    return precision, recall

def ap_all_point(precision, recall):
    """PASCAL VOC 2010+ all-point interpolated AP."""
    r = np.concatenate([[0.0], recall, [1.0]]); p = np.concatenate([[0.0], precision, [0.0]])
    p = np.maximum.accumulate(p[::-1])[::-1]                # precision envelope
    i = np.where(r[1:] != r[:-1])[0]
    return float(np.sum((r[i + 1] - r[i]) * p[i + 1]))

def ap_coco_101(precision, recall):
    """COCO-style AP: interpolated precision at 101 recall points."""
    p_env = np.maximum.accumulate(precision[::-1])[::-1] if len(precision) else precision
    out = []
    for rt in np.linspace(0, 1, 101):
        m = recall >= rt
        out.append(p_env[m].max() if m.any() else 0.0)
    return float(np.mean(out))

# Hand-checked example: 3 GT; detections ranked TP, FP, TP
p_ex, r_ex = pr_curve(np.array([True, False, True]), 3)
assert np.allclose(r_ex, [1/3, 1/3, 2/3]) and np.allclose(p_ex, [1, 0.5, 2/3])
assert abs(ap_all_point(p_ex, r_ex) - (1/3 * 1 + 1/3 * 2/3)) < 1e-12        # = 0.5556
print(f"Hand example: AP (all-point) = {ap_all_point(p_ex, r_ex):.4f} (expected 0.5556); "
      f"COCO-101 = {ap_coco_101(p_ex, r_ex):.4f}")

### A synthetic evaluation set

To compute meaningful PR curves we need many images with ground truth. We simulate a detector on 300 synthetic "images": each contains 1–6 objects of random size; the simulated detector finds small objects less often, localises them less precisely and also produces false positives.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements of a real detector.

In [ ]:
def simulate_eval_set(n_images=300, seed=7):
    r = np.random.default_rng(seed)
    gts, dets, sizes = {}, [], {}
    for im in range(n_images):
        n = r.integers(1, 7)
        side = np.exp(r.uniform(np.log(12), np.log(250), n))           # object size in px (12..250)
        x1 = r.uniform(0, 1000, n); y1 = r.uniform(0, 600, n)
        g = np.stack([x1, y1, x1 + side * 0.6, y1 + side], 1)          # pedestrian-like aspect ratio
        gts[im] = g; sizes[im] = np.sqrt((g[:, 2] - g[:, 0]) * (g[:, 3] - g[:, 1]))
        for k in range(n):
            p_detect = 1 / (1 + np.exp(-(side[k] - 30) / 10))          # small objects are missed more often
            if r.random() < p_detect:
                w, h = g[k, 2] - g[k, 0], g[k, 3] - g[k, 1]
                noise = r.normal(0, 0.08 + 1.5 / side[k], 4) * np.array([w, h, w, h])
                score = float(np.clip(r.normal(0.55 + 0.35 * min(side[k], 150) / 150, 0.12), 0.01, 0.99))
                dets.append((im, score, g[k] + noise))
        for _ in range(r.poisson(0.8)):                                # false positives
            s = np.exp(r.uniform(np.log(12), np.log(150))); x, y = r.uniform(0, 1000), r.uniform(0, 600)
            dets.append((im, float(np.clip(r.normal(0.35, 0.15), 0.01, 0.99)), np.array([x, y, x + 0.6 * s, y + s])))
    return gts, dets, sizes

gts, dets, sizes = simulate_eval_set()
n_gt = sum(len(v) for v in gts.values())
print(f"Synthetic set: {len(gts)} images, {n_gt} ground-truth objects, {len(dets)} detections")

scores50, tp50, _ = match_detections(dets, gts, 0.5)
p50, r50 = pr_curve(tp50, n_gt)
ap50 = ap_all_point(p50, r50)
iou_thrs = np.round(np.arange(0.5, 0.96, 0.05), 2)
ap_by_thr = []
for t in iou_thrs:
    _, tp_t, _ = match_detections(dets, gts, t)
    pp, rr = pr_curve(tp_t, n_gt)
    ap_by_thr.append(ap_coco_101(pp, rr))
ap_5095 = float(np.mean(ap_by_thr))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(r50, p50, color=OKABE_ITO[4], label=f'IoU 0.5 (AP = {ap50:.3f})')
_, tp75, _ = match_detections(dets, gts, 0.75); p75, r75 = pr_curve(tp75, n_gt)
axes[0].plot(r75, p75, color=OKABE_ITO[5], label=f'IoU 0.75 (AP = {ap_all_point(p75, r75):.3f})')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision'); axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.02)
axes[0].set_title('Precision-recall curves (synthetic detector)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(iou_thrs, ap_by_thr, 'o-', color=OKABE_ITO[2])
axes[1].set_xlabel('IoU threshold'); axes[1].set_ylabel('AP (COCO 101-point)')
axes[1].set_title(f'AP vs IoU threshold; AP@[.5:.95] = {ap_5095:.3f}'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"AP@0.5 (all-point) = {ap50:.3f}; AP@[.5:.95] (COCO) = {ap_5095:.3f}")
print(f"Maximum recall reached at IoU 0.5: {r50.max():.3f} -> {n_gt - int(tp50.sum())} objects are never detected at any threshold.")
print("AP falls as the IoU threshold rises because the same detections are judged with stricter localisation.")

In [ ]:
# AP by object size (COCO convention: small < 32^2 px^2, medium < 96^2, large otherwise)
def size_bin(s): return 'small' if s < 32 else ('medium' if s < 96 else 'large')
res = []
for name in ['small', 'medium', 'large']:
    g_sub = {im: g[[size_bin(s) == name for s in sizes[im]]] for im, g in gts.items()}
    n_sub = sum(len(v) for v in g_sub.values())
    # detections are attributed to the bin of the GT they overlap best (unmatched detections: by own size)
    d_sub = []
    for im, s, b in dets:
        g = gts[im]; ov = iou_matrix([b], g)[0]
        ref = sizes[im][np.argmax(ov)] if ov.max() > 0.1 else np.sqrt((b[2] - b[0]) * (b[3] - b[1]))
        if size_bin(ref) == name: d_sub.append((im, s, b))
    _, tp_s, _ = match_detections(d_sub, g_sub, 0.5)
    pp, rr = pr_curve(tp_s, n_sub)
    res.append({'size': name, 'GT objects': n_sub, 'AP@0.5': round(ap_all_point(pp, rr), 3),
                'max recall': round(float(rr.max()) if len(rr) else 0.0, 3)})
size_df = pd.DataFrame(res); display(size_df)
for r_ in res:
    print(f"  {r_['size']:6s}: {r_['GT objects']:4d} GT objects, AP@0.5 = {r_['AP@0.5']:.3f}, max recall = {r_['max recall']:.3f}")
worst = size_df.loc[size_df['AP@0.5'].idxmin(), 'size']
print(f"Lowest AP for {worst} objects -> distant pedestrians and children (few pixels) are the hardest case.")

**Why mAP is not a safety metric on its own:** it averages over classes, sizes and confidence thresholds, and treats a missed pedestrian at 10 m like a missed traffic cone at 80 m. Safety arguments need **targeted** metrics: recall for safety-relevant classes, per distance band and per condition, at the *chosen operating threshold* — see [05 Vulnerable road user detection](05_vulnerable_road_user_detection.ipynb).

---
## 7. Throughput and latency

- **Throughput (FPS)**: how many frames per second the detector can process (can be raised by batching or pipelining).
- **Latency**: the time from photons hitting the sensor to an actuator command. Camera rates are typically 10–30 Hz (nuScenes: 12 Hz), and the end-to-end perception→actuation budget is of the order of 100 ms. **FPS ≠ latency.**

During 100 ms at 100 km/h the vehicle travels 2.8 m. The benchmark below runs only when Ultralytics is available.

In [ ]:
def benchmark(model_name, image, n_runs=10, warmup=3):
    m = YOLO(model_name)
    bgr = np.ascontiguousarray(image[..., ::-1])
    for _ in range(warmup):
        m.predict(bgr, verbose=False)
    times, speeds = [], []
    for _ in range(n_runs):
        t0 = time.perf_counter(); r = m.predict(bgr, verbose=False)[0]
        times.append((time.perf_counter() - t0) * 1000); speeds.append(r.speed)
    n_params = sum(p.numel() for p in m.model.parameters()) / 1e6
    sp = {k: np.mean([s[k] for s in speeds]) for k in speeds[0]}
    return {'model': model_name, 'params (M)': round(n_params, 2), 'mean latency (ms)': round(np.mean(times), 1),
            'std (ms)': round(np.std(times), 1), 'throughput (FPS)': round(1000 / np.mean(times), 1),
            'pre/infer/post (ms)': f"{sp['preprocess']:.1f}/{sp['inference']:.1f}/{sp['postprocess']:.1f}"}

bench = []
if model is not None:
    import torch
    print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'} (results are hardware-specific)")
    for name in ['yolov8n.pt', 'yolo11n.pt', 'yolo26n.pt']:
        try:
            bench.append(benchmark(name, img))
        except Exception as e:
            print(f"  {name}: skipped ({type(e).__name__}: {str(e)[:60]})")
if bench:
    bench_df = pd.DataFrame(bench); display(bench_df)
    fastest = bench_df.loc[bench_df['mean latency (ms)'].idxmin(), 'model']
    print(f"Fastest on this machine: {fastest}.")
    for _, r in bench_df.iterrows():
        verdict = 'keeps up with' if r['mean latency (ms)'] < 1000 / 30 else 'does NOT keep up with'
        print(f"  {r['model']}: {r['mean latency (ms)']} ms -> {verdict} a 30 Hz camera on this hardware")
else:
    print("Benchmark skipped (Ultralytics, weights or image unavailable).")
v_kmh = np.array([30, 50, 100, 130]); lat = 0.1
for v in v_kmh:
    print(f"  {v:3d} km/h: {v/3.6*lat:4.1f} m travelled during a {lat*1000:.0f} ms latency")

---
## Exercise: analyse failure cases

Run the detector on your own street images (night, rain, fog, occlusion, unusual objects) and record false positives, false negatives and misclassifications. Why did each occur?

In [ ]:
# TODO: test your own images, e.g.
# my_img = load_image('https://...your-image-url...')
# if model is not None and my_img is not None:
#     r = model.predict(np.ascontiguousarray(my_img[..., ::-1]), conf=0.25, verbose=False)[0]
#     my_dets = [[model.names[int(c)], float(s), [float(v) for v in b]] for c, s, b in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxy)]
#     fig, ax = plt.subplots(figsize=(8, 8)); draw(ax, my_img, my_dets, 0.25, 'My image'); plt.show()
print("Suggested challenge images: night scenes, rain/fog, occlusions, crowds, unusual objects (e.g. a person in a wheelchair, an animal).")

---
## Safety perspective

- A detector's **confidence is not a calibrated probability**: a score of 0.9 does not mean a 90 % chance of being correct. Calibration is measured and corrected in [06_AI_Safety_ISO_PAS_8800/04 — Calibration](../06_AI_Safety_ISO_PAS_8800/04_calibration.ipynb).
- Missed or mislocalised objects caused by the model's limitations — small objects, crowds suppressed by NMS, out-of-distribution appearance — are **performance insufficiencies** (ISO 21448:2022); when they originate in the ML model or its training data, ISO/PAS 8800:2024 asks for data, training and verification measures specific to AI.
- The operating threshold and NMS parameters are **safety requirements**, not tuning details: they must be derived from the hazard analysis and verified per class, distance and condition.
- Both errors matter: a missed pedestrian can cause a collision; a phantom object can cause unnecessary emergency braking and a rear-end collision.

Continue with: [05 Vulnerable road user detection](05_vulnerable_road_user_detection.ipynb) · [02_Failures_and_Robustness/03 — OOD detection](../02_Failures_and_Robustness/03_out_of_distribution_detection.ipynb) · [04_SOTIF_ISO21448/01 — SOTIF fundamentals](../04_SOTIF_ISO21448/01_sotif_fundamentals.ipynb).

## Key takeaways

- A 2D detector outputs boxes, classes and confidence scores; box formats differ (corner `xyxy` vs COCO `xywh`) and must not be mixed.
- YOLOv8 (2023) is a baseline; YOLO11 (2024) and the NMS-free YOLO26 (Jan 2026) are current Ultralytics models (as of Oct 2026); always benchmark on the target hardware.
- IoU quantifies overlap; narrow objects such as pedestrians lose IoU quickly under small localisation errors.
- Greedy NMS removes duplicates but can also suppress real, closely spaced objects (crowds, child next to adult).
- AP is the area under the interpolated PR curve; AP@[.5:.95] averages over IoU thresholds and penalises poor localisation; small objects have the lowest AP.
- Throughput (FPS) is not latency; 100 ms at 100 km/h is 2.8 m of travel.
- Confidence is not calibrated probability; thresholds and NMS settings are safety-relevant design decisions (ISO 21448, ISO/PAS 8800).

## Self-assessment

1. What three outputs does an object detector provide for each object, and what are the two common box formats?
2. Compute the IoU of boxes [0, 0, 10, 10] and [5, 5, 15, 15].
3. Why does lowering the confidence threshold increase recall but decrease precision?
4. Explain how NMS can cause a false negative in a crowded scene, and one way to mitigate it.
5. A detector reaches AP@0.5 = 0.80 but AP@[.5:.95] = 0.45. What does this tell you about its localisation?
6. A detector runs at 80 FPS when frames are batched. Does it necessarily meet a 100 ms end-to-end latency budget? Why?

<details>
<summary>Show answers</summary>

1. Bounding box, class label and confidence score. Corner format [x1, y1, x2, y2] and COCO format [x, y, w, h].
2. Intersection 5 × 5 = 25; union 100 + 100 − 25 = 175; IoU = 25/175 ≈ 0.143.
3. More low-confidence detections are kept: some are real objects (more TPs, higher recall) but more are false (more FPs, lower precision).
4. Two people close together produce boxes with IoU above the NMS threshold, so the lower-scoring person is suppressed. Mitigations: class-specific or higher NMS thresholds for persons, Soft-NMS, NMS-free detectors, or fusion with LiDAR/radar evidence.
5. It usually finds the objects (good at loose IoU 0.5) but its boxes are not tight: precision drops quickly at stricter IoU thresholds.
6. No. Throughput with batching says nothing about the time from capture to output for a single frame; pre-processing, queuing, inference, post-processing, tracking, planning and actuation all add to latency.

</details>

## References

1. Redmon, J., Divvala, S., Girshick, R., Farhadi, A. (2016). You Only Look Once: Unified, Real-Time Object Detection. CVPR. arXiv:1506.02640
2. Ren, S., He, K., Girshick, R., Sun, J. (2015). Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks. NeurIPS. arXiv:1506.01497
3. Carion, N. et al. (2020). End-to-End Object Detection with Transformers (DETR). ECCV. arXiv:2005.12872
4. Zhao, Y. et al. (2024). DETRs Beat YOLOs on Real-time Object Detection (RT-DETR). CVPR. arXiv:2304.08069
5. Lin, T.-Y. et al. (2014). Microsoft COCO: Common Objects in Context. ECCV. arXiv:1405.0312
6. Everingham, M. et al. (2010). The PASCAL Visual Object Classes (VOC) Challenge. *IJCV* 88, 303–338. https://doi.org/10.1007/s11263-009-0275-4
7. Ultralytics (2026). YOLO26 vs YOLO11 comparison. https://docs.ultralytics.com/compare/yolo26-vs-yolo11
8. ISO 21448:2022. *Road vehicles — Safety of the intended functionality.* https://www.iso.org/standard/77490.html
9. ISO/PAS 8800:2024. *Road vehicles — Safety and artificial intelligence.*
10. Patel, M., Jung, R. (2026). Uncertainty Evaluation to Support Safety of the Intended Functionality Analysis for Identifying Performance Insufficiencies in ML-Based LiDAR Object Detection. VEHITS 2026. https://doi.org/10.5220/0014967200004030

### Further practice

Companion hands-on material: [AI Perception Safety Workshop — Notebook 02, Object detection basics](https://github.com/milinpatel07/AI-Perception-Safety-Workshop/blob/main/notebooks/02_object_detection_basics.ipynb).

---
**Navigation:** ← [03 LiDAR and point clouds](03_lidar_point_clouds.ipynb) · [Module index](README.md) · [05 Vulnerable road user detection](05_vulnerable_road_user_detection.ipynb) →